# Project 1 — Monthly Analytics-Ready Dataset for **Grameenphone (DSE: GP)**
### FIN 4333 · Financial Analytics — Team Hemisphere


## 0 · Install the libraries (run once)

**Before:** This single cell installs the third-party packages the notebook imports. The standard-library
Modules (`os`, `json`, `pathlib`, `datetime`) need no installation, so only five packages are listed.

In [11]:
# %pip installs into THIS kernel's environment (avoids the "installed but ModuleNotFoundError" mismatch).
%pip install --quiet pandas numpy matplotlib requests python-docx
# pandas  -> data tables / cleaning / merging
# numpy   -> numeric helpers
# matplotlib -> the overview chart embedded in the memo
# requests   -> calls the live World Bank Indicators API
# python-docx -> writes the Word (.docx) project memo

Note: you may need to restart the kernel to use updated packages.


**After:** running this once makes every later `import` succeed inside the active kernel. Using the
`%pip` *magic* (rather than `!pip`) guarantees the packages land in the same interpreter that executes the
notebook, which is the usual cause of "I installed it but it can't be found." If a package was upgraded you
may be prompted to restart the kernel; on a fresh environment you can continue straight to Section 1. This
cell does no data work — it only prepares the environment so the pipeline below is reproducible on any machine.

## 1 · Setup — imports, configuration, and project paths

**Before:** we import the libraries, put every tunable choice (company, ticker, date window) in one place,
and locate the project root so all file paths resolve no matter where the notebook is launched from.

In [12]:
# --- Core libraries -------------------------------------------------------
import os, json                                   # os: paths; json: read/write the World Bank cache
from pathlib import Path                           # Path: robust, OS-independent file paths
from datetime import date                          # date: stamp the source log with today's date
import numpy as np                                 # numpy: numeric helpers (NaN handling, arrays)
import pandas as pd                                # pandas: the main data-wrangling library
import matplotlib                                  # matplotlib: plotting backend for the chart
matplotlib.use("Agg")                              # "Agg" = headless backend (saves PNG without a screen)
import matplotlib.pyplot as plt                    # pyplot: the plotting interface we actually call
try:
    import requests                                # requests: HTTP client for the live World Bank API
except Exception:
    requests = None                                # if unavailable, the macro step uses the cache instead
try:
    import docx                                     # python-docx: build the .docx memo
except ImportError:                                 # if missing, install it on the fly
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "python-docx", "-q"])
    import docx

# --- Configuration (change these to re-point the pipeline at another company) ---
COMPANY_NAME   = "Grameenphone Ltd."               # appears in every row of the final dataset
TICKER         = "GP"                              # DSE code; also used to build file names
START, END     = "2020-01-31", "2024-12-31"        # inclusive month-end window (60 months)
STOCK_DATE_FMT = "%m/%d/%Y"                         # investing.com exports dates as MM/DD/YYYY

# --- Locate the project root: walk upward until we find the raw_data/ folder ---
def find_root(start=None):                          # makes the notebook runnable from any working dir
    p = Path(start or Path.cwd()).resolve()         # start from the current directory
    for cand in [p, *p.parents]:                    # check this folder, then each parent
        if (cand / "raw_data").is_dir():            # the root is the one that contains raw_data/
            return cand
    raise FileNotFoundError("Could not locate project root (a folder containing raw_data/).")
ROOT = find_root()                                  # ROOT is now the project's top folder

# --- Derived paths for inputs and outputs --------------------------------
STOCK_FILE    = ROOT / "raw_data" / "stock" / "Grameenphone_Stock_Price_History.csv"  # raw daily prices
FIRM_FILE     = ROOT / "raw_data" / "firm_reports" / "gp_annual_firm_data.csv"        # annual firm variables
MACRO_DIR     = ROOT / "raw_data" / "macro_api_raw"                                   # World Bank raw JSON
PROCESSED_DIR = ROOT / "processed_data"; FINAL_DIR = ROOT / "final_data"              # intermediate + final
LOGS_DIR      = ROOT / "logs";           OUT_DIR   = ROOT / "outputs"                 # logs + deliverables
for d in (PROCESSED_DIR, FINAL_DIR, LOGS_DIR, OUT_DIR):
    d.mkdir(parents=True, exist_ok=True)            # create output folders if they don't exist yet

# --- World Bank indicators: API code -> friendly column name -------------
WB_COUNTRY    = "BD"                                                       # ISO code for Bangladesh
WB_INDICATORS = {"NY.GDP.MKTP.KD.ZG": "bd_gdp_growth_annual_pct",          # GDP growth, annual %
                 "FP.CPI.TOTL.ZG":    "bd_inflation_annual_pct"}           # CPI inflation, annual %

# --- The canonical 60-row month-end grid (one row per month) -------------
MONTH_INDEX   = pd.date_range(START, END, freq="ME")  # "ME" = month-end dates; 60 of them
print(f"Project root: {ROOT}")
print(f"Window: {MONTH_INDEX.min():%Y-%m} .. {MONTH_INDEX.max():%Y-%m}  ({len(MONTH_INDEX)} months)")

Project root: C:\Users\Golam Rahat\Project_1a_Team_Hemisphere
Window: 2020-01 .. 2024-12  (60 months)


**After:** this establishes the spine of the whole pipeline. Centralising the configuration means
re-using the notebook for a different company is a small, auditable edit instead of a hunt through the code.
`find_root()` keeps the notebook portable (it works whether you launch it from `notebooks/` or the project
root). Most importantly, `MONTH_INDEX` is the canonical list of 60 month-end dates: every later step is
aligned to it, which is exactly what guarantees the final dataset has **one row per month with no gaps or
duplicates**.

## 2 · Stock — import and clean the raw daily price file

**Before:** the investing.com export is newest-row-first with `MM/DD/YYYY` dates and columns
`Date, Price, Open, High, Low, Vol., Change %`. We reduce it to two trustworthy columns — a parsed `date`
and a numeric `close` (their `Price`) — and defensively drop anything unusable.

In [13]:
raw = pd.read_csv(STOCK_FILE)                                          # read the raw daily export
print("Raw columns:", list(raw.columns), "| raw rows:", len(raw))     # show what we received

# Auto-detect the DATE column and the CLOSING-PRICE column (portals name them differently).
date_col  = next(c for c in raw.columns if c.lower() in ("date", "trade date", "time"))   # find the date col
price_pref = ["adj close", "adj. close", "close", "price", "ltp", "closing price", "close*"]  # close synonyms
close_col = next((c for c in raw.columns if c.lower() in price_pref),                      # first matching name
                 raw.select_dtypes("number").columns[0])                                   # else first numeric col

daily = raw[[date_col, close_col]].rename(columns={date_col: "date", close_col: "close"})  # keep + rename 2 cols
daily["date"]  = pd.to_datetime(daily["date"], format=STOCK_DATE_FMT, errors="coerce")     # parse US dates -> NaT if bad
daily["close"] = pd.to_numeric(daily["close"].astype(str)                                  # treat as text first
                               .str.replace(",", "", regex=False),                         # "1,234.5" -> "1234.5"
                               errors="coerce")                                            # non-numeric -> NaN
daily = (daily.dropna(subset=["date", "close"])      # drop rows whose date or price could not be parsed
              .loc[daily["close"] > 0]               # keep only positive prices (a price must be > 0)
              .drop_duplicates(subset="date")        # one record per trading day
              .sort_values("date")                   # put rows in chronological order
              .reset_index(drop=True))               # tidy the row index after sorting
print(f"Cleaned daily rows: {len(daily)} (date='{date_col}', close='{close_col}')")
print(f"Span: {daily['date'].min():%Y-%m-%d} .. {daily['date'].max():%Y-%m-%d}")           # confirm coverage
daily.head()                                                                               # peek at the result

Raw columns: ['Date', 'Price', 'Open', 'High', 'Low', 'Vol.', 'Change %'] | raw rows: 1163
Cleaned daily rows: 1163 (date='Date', close='Price')
Span: 2020-01-01 .. 2024-12-30


,date,close
0,2020-01-01,281.4
1,2020-01-02,282.9
2,2020-01-05,282.4
3,2020-01-06,275.0
4,2020-01-07,272.0


**After:** the messy export is now a tidy, ascending daily series of `date` and positive `close`
(~1,163 trading days spanning Jan 2020 – Dec 2024). Each cleaning step removes a specific real-world hazard:
explicit `%m/%d/%Y` parsing avoids day/month ambiguity; `errors="coerce"` + `dropna` strip header artefacts
and blank rows; comma-stripping fixes thousands separators; the positive-price filter rejects bad ticks; and
de-duplicating + sorting produce the ordered series that a correct month-end resample depends on. This clean
daily series is the raw material for the monthly price and return computed next.

## 3 · Stock — month-end price and monthly return

**Before:** the project needs monthly data, so we take the **last close of each calendar month**, force the
full 60-month grid, and compute the **monthly return from month-end prices**: $r_t = P_t/P_{t-1}-1$. Because
the DSE was closed for all of April 2020, that month has no price and must stay blank — not be filled in.

In [14]:
monthly_close = (daily.set_index("date")["close"]      # index the daily closes by their date
                       .resample("ME").last()           # take the LAST close in each calendar month ("ME"=month-end)
                       .reindex(MONTH_INDEX))            # force exactly the 60 canonical month-end dates
monthly = pd.DataFrame({"date": MONTH_INDEX,             # canonical month-end dates (the spine)
                        "month_end_price": monthly_close.values})  # the month-end close for each month

# Monthly simple return from month-end prices. fill_method=None is CRITICAL:
# without it, pandas forward-fills the missing April-2020 price and invents a fake 0% return.
monthly["monthly_stock_return"] = monthly["month_end_price"].pct_change(fill_method=None)

monthly.to_csv(PROCESSED_DIR / "monthly_stock.csv", index=False)   # save the processed monthly series
gap = monthly.loc[monthly["month_end_price"].isna(), "date"].dt.strftime("%Y-%m").tolist()  # months with no price
print(f"Monthly rows: {len(monthly)} | months with no price: {gap}  (Apr-2020 = DSE COVID closure)")
monthly.head(7)                                                    # show Jan-2020 .. Jul-2020 incl. the gap

Monthly rows: 60 | months with no price: ['2020-04']  (Apr-2020 = DSE COVID closure)


,date,month_end_price,monthly_stock_return
0,2020-01-31,257.8,NaN
1,2020-02-29,274.4,0.064391
2,2020-03-31,238.8,-0.129738
3,2020-04-30,NaN,NaN
4,2020-05-31,255.9,NaN
5,2020-06-30,238.8,-0.066823
6,2020-07-31,258.5,0.082496


**After:** the daily series collapses to a 60-row monthly series. `resample("ME").last()` is the
standard, defensible month-end convention, and `reindex(MONTH_INDEX)` guarantees no month is missing or
duplicated. The `fill_method=None` argument is the key correctness detail: April 2020 genuinely had no
trading, so its price stays `NaN` and its return is `NaN` rather than a misleading 0%. That also makes May
2020's return `NaN` (its prior month-end is missing) and Jan 2020's `NaN` (no prior month) — three explained
blanks in total. This `month_end_price`/`monthly_stock_return` pair is the market-data half of the final dataset.

## 4 · Firm — load the annual firm variables

**Before:** Revenue, Total Assets and EPS were read from Grameenphone's audited Annual Reports 2020–2024
(each year's primary statements, cross-checked against the five-year summary in AR2024, Table-1, p.89) into
`gp_annual_firm_data.csv`. Here we load that file and standardise its column names.

In [15]:
firm_raw = pd.read_csv(FIRM_FILE)                                   # read the firm CSV (incl. a source_note column)
firm = firm_raw.rename(columns={"annual_revenue_mn_bdt":      "annual_revenue",       # -> shorter names
                                "annual_total_assets_mn_bdt": "annual_total_assets",  #    used by the pipeline
                                "annual_eps_bdt":             "annual_eps"})
firm = firm[["year", "annual_revenue", "annual_total_assets", "annual_eps"]].copy()   # keep the 4 needed columns
firm.to_csv(PROCESSED_DIR / "firm_annual_clean.csv", index=False)  # save the cleaned annual table
firm                                                               # display the 5 annual rows

,year,annual_revenue,annual_total_assets,annual_eps
0,2020,139606,148184,27.54
1,2021,143066,163007,25.28
2,2022,150403,185087,22.29
3,2023,158716,200420,24.49
4,2024,158447,198853,26.89


**After:** we now have a clean annual table keyed by `year` (2020–2024) with the three required firm
variables. Revenue and total assets are in **million BDT**; EPS is **BDT per share** (face value BDT 10,
1,350 million shares). Keeping only these columns and a clean `year` key is what lets the alignment step in
Section 5 broadcast each year's figures onto the right monthly rows.

## 5 · Macro — Bangladesh indicators from the **World Bank Indicators API**

**Before:** we request two indicators for Bangladesh, 2020–2024 — `NY.GDP.MKTP.KD.ZG` (GDP growth) and
`FP.CPI.TOTL.ZG` (CPI inflation). The code calls the **live API** and saves the **raw JSON**; if the call
fails (e.g. no internet) it falls back to the cached raw files so the notebook still completes.

In [16]:
rows, used_live = [], False                                        # collect observations; track if API was reached
for code_id, friendly in WB_INDICATORS.items():                    # loop over the two indicators
    url = (f"https://api.worldbank.org/v2/country/{WB_COUNTRY}/indicator/{code_id}"   # build the REST URL
           f"?format=json&per_page=100&date=2020:2024")                              # JSON, all years 2020-2024
    cache = MACRO_DIR / ("worldbank_gdp_growth_raw.json" if code_id.startswith("NY") # matching cache file name
                         else "worldbank_inflation_raw.json")
    payload = None                                                 # will hold the parsed JSON response
    if requests is not None:                                       # only try the network if requests imported
        try:
            resp = requests.get(url, timeout=20); resp.raise_for_status()  # GET; raise on HTTP error
            payload = resp.json()                                          # parse JSON
            if isinstance(payload, list) and len(payload) == 2 and payload[1]:  # valid WB shape = [meta, data]
                cache.write_text(json.dumps(payload, indent=2)); used_live = True  # save fresh raw + mark live
            else:
                payload = None                                     # odd/empty response -> fall back to cache
        except Exception as e:
            print(f"  live API failed for {code_id} ({e}); using cached raw JSON")  # explain the fallback
            payload = None
    if payload is None:                                            # fallback path: read the cached raw JSON
        payload = json.loads(cache.read_text())
    for obs in payload[1]:                                         # payload[1] is the list of yearly observations
        if obs.get("value") is not None:                          # skip years with no reported value
            rows.append({"year": int(obs["date"]), friendly: float(obs["value"])})  # (year, indicator) record

macro = (pd.DataFrame(rows).groupby("year").first()               # collapse the two indicators onto one row/year
                           .reset_index().sort_values("year").reset_index(drop=True))
macro.to_csv(PROCESSED_DIR / "macro_annual_clean.csv", index=False)  # save the cleaned macro table
print(f"Live API used this run: {used_live}")                     # True on a connected machine, False if cached
macro                                                             # display the 5 annual macro rows

Live API used this run: True


,year,bd_gdp_growth_annual_pct,bd_inflation_annual_pct
0,2020,3.448018,5.691075
1,2021,6.938679,5.545654
2,2022,7.099829,7.696954
3,2023,5.775112,9.883503
4,2024,4.223259,10.465748


**After:** we now have a tidy annual macro table keyed by `year`, and the raw API response is saved in
`raw_data/macro_api_raw/` as evidence of the retrieval. The live/cached design makes the run reproducible
either way: online it refreshes from the World Bank (so the values stay current — e.g. revised figures flow
through automatically), and offline it uses the saved JSON. Note the World Bank reports Bangladesh GDP on a
fiscal-year (Jul–Jun) basis, so each year's growth figure is a fiscal-year value mapped to the calendar year.
These two columns are the macro half of the final dataset.

## 6 · Align and merge — assemble the final monthly dataset

**Before:** the dataset is monthly but the firm and macro variables are annual. The **alignment rule** is to
assign each annual value to **every month of the same calendar year**. We add identifier and `year`/`month`
columns, then **left-merge** the annual tables on `year`, and order the required columns.

In [17]:
final = monthly.copy()                                             # start from the monthly stock series
final.insert(0, "company_name", COMPANY_NAME)                      # constant company name (column 1)
final.insert(1, "ticker_or_code", TICKER)                          # constant ticker (column 2)
final["year"]  = final["date"].dt.year                             # calendar year -> the alignment key
final["month"] = final["date"].dt.month                            # calendar month 1-12 (for convenience)
final = final.merge(firm,  on="year", how="left")                  # broadcast firm annual values onto months
final = final.merge(macro, on="year", how="left")                  # broadcast macro annual values onto months
final = final[["company_name", "ticker_or_code", "date", "year", "month",   # fix the required column order
               "month_end_price", "monthly_stock_return",
               "annual_revenue", "annual_total_assets", "annual_eps",
               "bd_inflation_annual_pct", "bd_gdp_growth_annual_pct"]]
print("Final shape:", final.shape)                                 # expect (60, 12)
final.head(7)                                                      # preview including the Apr-2020 gap

Final shape: (60, 12)


,company_name,ticker_or_code,date,year,month,month_end_price,monthly_stock_return,annual_revenue,annual_total_assets,annual_eps,bd_inflation_annual_pct,bd_gdp_growth_annual_pct
0,Grameenphone Ltd.,GP,2020-01-31,2020,1,257.8,NaN,139606,148184,27.54,5.691075,3.448018
1,Grameenphone Ltd.,GP,2020-02-29,2020,2,274.4,0.064391,139606,148184,27.54,5.691075,3.448018
2,Grameenphone Ltd.,GP,2020-03-31,2020,3,238.8,-0.129738,139606,148184,27.54,5.691075,3.448018
3,Grameenphone Ltd.,GP,2020-04-30,2020,4,NaN,NaN,139606,148184,27.54,5.691075,3.448018
4,Grameenphone Ltd.,GP,2020-05-31,2020,5,255.9,NaN,139606,148184,27.54,5.691075,3.448018
5,Grameenphone Ltd.,GP,2020-06-30,2020,6,238.8,-0.066823,139606,148184,27.54,5.691075,3.448018
6,Grameenphone Ltd.,GP,2020-07-31,2020,7,258.5,0.082496,139606,148184,27.54,5.691075,3.448018


**After:** this produces the 60×12 analytics-ready frame. Because the firm and macro tables each have
exactly one row per `year`, a left-merge on `year` repeats every annual value across that year's twelve
monthly rows — which is precisely the alignment rule the brief requires (e.g. all twelve 2023 rows carry the
2023 revenue, assets, EPS, inflation and GDP-growth values). The result combines the monthly market data with
the broadcast annual data into the single table that is the project's main deliverable.

## 7 · Quality checks

**Before:** we verify structure (60 rows, one per month, continuous month-end dates), missing values,
duplicates, variable types, and suspicious values. The April-2020 price gap and the three explained `NaN`
returns are checked **explicitly**, so they register as understood rather than as silent holes. Results are
saved to `logs/quality_check_sheet.csv`.

In [18]:
checks = []                                                        # collect one dict per check
def add(name, passed, detail):                                     # helper: record a PASS/REVIEW row
    checks.append({"check": name, "result": "PASS" if passed else "REVIEW", "detail": detail})

add("Row count = 60 months", len(final) == 60, f"rows={len(final)}")                       # exactly 60 rows
add("One row per month (no dup dates)", final["date"].is_unique, f"unique={final['date'].nunique()}")  # no dups
add("Dates month-end Jan-2020..Dec-2024", list(final['date']) == list(MONTH_INDEX), "vs expected grid")  # right dates
add("Continuous monthly spacing (no gaps)",
    final["date"].diff().dropna().dt.days.between(28, 31).all(), "spacing 28-31 days")     # consecutive months

# April-2020 has no month-end price because the DSE was closed 26 Mar - 31 May 2020 (COVID-19).
price_missing = set(final.loc[final["month_end_price"].isna(), "date"])                    # which months lack a price
add("month_end_price present (except documented DSE closure)",
    price_missing == {pd.Timestamp("2020-04-30")},                                         # only April-2020 may be missing
    f"missing={sorted(d.strftime('%Y-%m') for d in price_missing)} (Apr-2020 DSE closure)")
ret_missing = set(final.loc[final["monthly_stock_return"].isna(), "date"])                 # which months lack a return
add("monthly_stock_return NaNs all explained",
    ret_missing == {pd.Timestamp("2020-01-31"), pd.Timestamp("2020-04-30"), pd.Timestamp("2020-05-31")},  # the 3 expected
    f"missing={sorted(d.strftime('%Y-%m') for d in ret_missing)} (start month + DSE closure)")
for c in ["annual_revenue","annual_total_assets","annual_eps",                             # annual cols must be complete
          "bd_inflation_annual_pct","bd_gdp_growth_annual_pct"]:
    add(f"Missing {c}", int(final[c].isna().sum()) == 0, f"n={int(final[c].isna().sum())}")

add("month_end_price > 0", (final["month_end_price"].dropna() > 0).all(),                  # prices are positive
    f"min={final['month_end_price'].min():.2f}")
add("No extreme monthly returns (|r|<0.50)", final["monthly_stock_return"].abs().dropna().lt(0.50).all(),  # sanity bound
    f"max|r|={final['monthly_stock_return'].abs().max():.4f}")
add("annual_eps in plausible band (0-100)", final["annual_eps"].between(0,100).all(),      # EPS sanity range
    f"range=[{final['annual_eps'].min()},{final['annual_eps'].max()}]")
add("Types correct (date/float/int)",                                                      # dtype check
    pd.api.types.is_datetime64_any_dtype(final["date"]) and
    pd.api.types.is_numeric_dtype(final["month_end_price"]) and
    pd.api.types.is_integer_dtype(final["year"]), "date=datetime, price=float, year=int")

qc = pd.DataFrame(checks)                                          # turn the list into a table
qc.to_csv(LOGS_DIR / "quality_check_sheet.csv", index=False)      # save the evidence sheet
print(f"{(qc.result=='PASS').sum()} PASS / {(qc.result=='REVIEW').sum()} REVIEW")          # headline tally
qc                                                                # display all checks

15 PASS / 0 REVIEW


,check,result,detail
0,Row count = 60 months,PASS,rows=60
1,One row per month (no dup dates),PASS,unique=60
2,Dates month-end Jan-2020..Dec-2024,PASS,vs expected grid
3,Continuous monthly spacing (no gaps),PASS,spacing 28-31 days
4,month_end_price present (except documented DSE...,PASS,missing=['2020-04'] (Apr-2020 DSE closure)
5,monthly_stock_return NaNs all explained,PASS,"missing=['2020-01', '2020-04', '2020-05'] (sta..."
6,Missing annual_revenue,PASS,n=0
7,Missing annual_total_assets,PASS,n=0
8,Missing annual_eps,PASS,n=0
9,Missing bd_inflation_annual_pct,PASS,n=0


**After:** data quality becomes an explicit, auditable table — every check should read **PASS**, with
the DSE-closure gap surfaced openly instead of hidden. The two missing-value checks are written to *expect*
the exact closure-related blanks, so they confirm the data matches the documented story rather than merely
counting nulls. The largest monthly move (~36%, Aug-2024) is a real swing that sits within sane bounds. This
sheet is one of the required deliverables and is what lets a reader trust the final dataset at a glance.

## 8 · Save the final dataset, the data dictionary, and the source log

**Before:** we write the headline CSV (`final_data/final_monthly_dataset.csv`, dates as `YYYY-MM-DD` text)
and regenerate the two CSV documentation deliverables — the **data dictionary** (what each column means) and
the **source log** (where each component came from) — so the documentation always matches the data produced.

In [19]:
final_out = final.copy()                                           # copy so we don't alter the in-memory frame
final_out["date"] = final_out["date"].dt.strftime("%Y-%m-%d")      # write dates as plain ISO text
final_out.to_csv(FINAL_DIR / "final_monthly_dataset.csv", index=False)   # <-- the main deliverable

# --- Mini data dictionary: one row per column of the final dataset -------
dd = [
 ("company_name","Name of the selected company","text","monthly","Firm report","Constant"),
 ("ticker_or_code","DSE ticker / company code","text","monthly","Stock source","Constant: GP"),
 ("date","Month-end calendar date","YYYY-MM-DD","monthly","Derived","Last day of month"),
 ("year","Calendar year","integer","monthly","Derived","Alignment key"),
 ("month","Calendar month (1-12)","integer","monthly","Derived",""),
 ("month_end_price","Last traded close of the month","BDT","monthly","Daily stock file ('Price')","NaN Apr-2020: DSE closed 26 Mar-31 May 2020"),
 ("monthly_stock_return","P_t/P_(t-1)-1 on month-end price","decimal","monthly","Derived","NaN Jan/Apr/May-2020 (start + DSE closure)"),
 ("annual_revenue","Annual revenue / net sales","million BDT","annual->monthly","GP annual report","Repeats within year"),
 ("annual_total_assets","Year-end total assets","million BDT","annual->monthly","GP annual report",""),
 ("annual_eps","Annual basic earnings per share","BDT/share","annual->monthly","GP annual report","Face value BDT 10"),
 ("bd_inflation_annual_pct","Inflation, consumer prices","annual %","annual->monthly","World Bank FP.CPI.TOTL.ZG","Bangladesh"),
 ("bd_gdp_growth_annual_pct","GDP growth","annual %","annual->monthly","World Bank NY.GDP.MKTP.KD.ZG","Bangladesh; fiscal-year basis"),
]
pd.DataFrame(dd, columns=["variable","Description","unit","frequency","source","notes"]).to_csv(OUT_DIR / "data_dictionary.csv", index=False)

# --- Source log: where each data component came from ---------------------
today = date.today().isoformat()                                   # ISO date stamp for the log
macro_note = "World Bank Indicators API (live call; cached raw JSON retained as offline fallback)"  # env-independent
src = [
 ("Stock - daily prices","investing.com - Grameenphone Historical Data (Download Data)","raw_data/stock/Grameenphone_Stock_Price_History.csv",today,"'Price'=close; 1,163 days; no trades Apr-2020 (DSE closed 26 Mar-31 May 2020)"),
 ("Firm - Revenue/Total Assets/EPS","Grameenphone Annual Reports 2020-2024 (audited)","raw_data/firm_reports/*.pdf + gp_annual_firm_data.csv",today,"Per-year primary statements; cross-checked vs AR2024 Table-1 (p.89)"),
 ("Macro - GDP growth","World Bank API NY.GDP.MKTP.KD.ZG (BD, 2020:2024)","raw_data/macro_api_raw/worldbank_gdp_growth_raw.json",today,macro_note),
 ("Macro - Inflation CPI","World Bank API FP.CPI.TOTL.ZG (BD, 2020:2024)","raw_data/macro_api_raw/worldbank_inflation_raw.json",today,macro_note),
]
pd.DataFrame(src, columns=["data_component","source","saved_as","retrieved_on","notes"]).to_csv(LOGS_DIR / "source_log.csv", index=False)
print("Saved final dataset, data dictionary, source log.")

Saved final dataset, data dictionary, source log.


**After:** the final dataset is now on disk in `final_data/`, accompanied by two documentation files
generated from the same run. The **data dictionary** explains every column's meaning, unit, frequency and
source (including the documented April-2020 gap), so a reader can interpret the table without guessing. The
**source log** records provenance for all three data components with today's date. Because these are
regenerated by the pipeline rather than hand-written, they cannot drift out of sync with the actual data.

## 9 · Build the overview chart and the Word project memo

**Before:** finally we render a two-panel overview chart (month-end price + monthly returns) and assemble the
**project memo as a `.docx`**, embedding the firm/macro table and the chart. This is the human-readable
summary deliverable.

In [20]:
# --- Overview chart -------------------------------------------------------
chart_path = OUT_DIR / "overview_chart.png"                        # where to save the PNG
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(9, 5.2), sharex=True, # two stacked panels sharing the x-axis
                               gridspec_kw={"height_ratios":[2,1]})  # price panel taller than the return panel
ax1.plot(final["date"], final["month_end_price"], color="#1f6f8b", linewidth=2)  # price line (gap shows at Apr-2020)
ax1.set_ylabel("Month-end price (BDT)"); ax1.grid(alpha=0.3)        # label + light grid
ax1.set_title(f"{COMPANY_NAME} (DSE: {TICKER}) - month-end price and monthly return, 2020-2024")
cols = ["#2a9d8f" if v >= 0 else "#e76f51" for v in final["monthly_stock_return"].fillna(0)]  # green up / red down
ax2.bar(final["date"], final["monthly_stock_return"]*100, width=20, color=cols)  # returns as % bars
ax2.set_ylabel("Monthly return (%)"); ax2.axhline(0, color="#555", linewidth=0.8); ax2.grid(alpha=0.3)  # zero line
fig.tight_layout(); fig.savefig(chart_path, dpi=150); plt.close(fig)  # save at 150 DPI and free memory

# --- Project memo (.docx) -------------------------------------------------
from docx import Document                                          # document object
from docx.shared import Pt, Inches, RGBColor                       # sizing/colour helpers
from docx.enum.text import WD_ALIGN_PARAGRAPH                      # paragraph alignment
NAVY = RGBColor(0x1F,0x3A,0x5F)                                    # heading colour
doc = Document()                                                   # new empty Word document
doc.styles["Normal"].font.name = "Calibri"; doc.styles["Normal"].font.size = Pt(11)  # base font
h = doc.add_heading("Project 1 Memo - Monthly Analytics-Ready Dataset", level=0); h.runs[0].font.color.rgb = NAVY
doc.add_paragraph().add_run("FIN 4333 Financial Analytics - Team Hemisphere").bold = True  # subtitle
span = f"{final['date'].min():%b %Y} - {final['date'].max():%b %Y}"                         # coverage string
doc.add_paragraph(f"Company: {COMPANY_NAME} (DSE: {TICKER})  |  Coverage: {span}  |  {len(final)} monthly rows")

doc.add_heading("1. Company and rationale", level=1)               # ---- section 1
doc.add_paragraph("Grameenphone is Bangladesh's largest listed telecom operator. It was chosen for its "
    "long, consistent audited annual reports, active DSE trading (a complete daily price history) and a "
    "January-December fiscal year that aligns with the project's calendar-year requirement.")
doc.add_heading("2. Data sources", level=1)                        # ---- section 2 (bulleted sources)
for label, text in [
    ("Stock (daily -> monthly)","Daily price history from investing.com (Grameenphone Historical Data), "
        "1 Jan 2020 - 30 Dec 2024 (1,163 trading days); 'Price' cleaned and resampled to month-end."),
    ("Firm (annual)","Revenue, Total Assets and EPS from Grameenphone's audited Annual Reports 2020-2024, "
        "read from each year's primary statements and cross-checked vs AR2024 Table-1 (p.89)."),
    ("Macro (annual)","Bangladesh GDP growth and CPI inflation from the World Bank Indicators API; the raw "
        "JSON response is saved, and a cached copy is retained as an offline fallback.")]:
    p = doc.add_paragraph(style="List Bullet"); p.add_run(label+": ").bold = True; p.add_run(text)

doc.add_heading("3. Firm and macro values used", level=1)          # ---- section 3 (a values table)
fm = firm.merge(macro, on="year", how="left").sort_values("year")  # one row per year, firm + macro side by side
hdr = ["Year","Revenue (mn BDT)","Total assets (mn BDT)","EPS (BDT)","Inflation %","GDP growth %"]
tbl = doc.add_table(rows=1, cols=len(hdr)); tbl.style = "Light Grid Accent 1"  # styled table
for i,htext in enumerate(hdr):                                     # write the header row
    tbl.rows[0].cells[i].text = htext; tbl.rows[0].cells[i].paragraphs[0].runs[0].bold = True
for _, r in fm.iterrows():                                         # write one row per year
    c = tbl.add_row().cells
    c[0].text=str(int(r["year"])); c[1].text=f"{int(r['annual_revenue']):,}"; c[2].text=f"{int(r['annual_total_assets']):,}"
    c[3].text=f"{r['annual_eps']:.2f}"; c[4].text=f"{r['bd_inflation_annual_pct']:.2f}"; c[5].text=f"{r['bd_gdp_growth_annual_pct']:.2f}"

doc.add_heading("4. Major steps", level=1)                         # ---- section 4 (numbered method)
for s in ["Import and clean the daily stock file (parse dates, coerce prices, drop bad/duplicate rows, sort).",
          "Resample daily closes to month-end prices; compute monthly returns from them.",
          "Load the annual firm variables (Revenue, Total Assets, EPS).",
          "Retrieve the two macro indicators from the World Bank API and save the raw JSON.",
          "Align annual firm and macro values onto monthly rows by calendar year.",
          "Merge all components into one monthly dataset and order the required columns.",
          "Run quality checks and save the final CSV."]:
    doc.add_paragraph(s, style="List Number")

doc.add_heading("5. Alignment rule", level=1)                      # ---- section 5
doc.add_paragraph("The dataset is monthly but firm and macro variables are annual, so each annual value is "
    "assigned to every month of the same calendar year (all twelve 2023 rows carry the 2023 figures). This is "
    "a left-merge on the 'year' key.")
doc.add_heading("6. Data-quality notes", level=1)                  # ---- section 6 (bulleted notes)
np_, nr_ = int((qc.result=='PASS').sum()), int((qc.result=='REVIEW').sum())
for s in [f"All {len(qc)} automated checks: {np_} PASS / {nr_} REVIEW.",
          "60/60 monthly rows; dates month-end and continuous (no gaps or duplicates).",
          "April 2020 has no month-end price because the DSE suspended all trading 26 Mar - 31 May 2020 "
          "(COVID-19; last session 25 Mar, resumed 31 May). The dataset preserves this true 'no trade' state.",
          "monthly_stock_return is NaN for Jan-2020 (no prior month), Apr-2020 (no price) and May-2020 "
          "(prior month NaN). A gap-free series could carry forward the 25-Mar-2020 close - a modelling choice.",
          "Annual firm/macro columns fully populated after alignment.",
          "World Bank reports Bangladesh GDP on a fiscal-year (Jul-Jun) basis, so each year's growth figure "
          "is a fiscal-year value mapped onto the calendar year."]:
    doc.add_paragraph(s, style="List Bullet")
doc.add_heading("7. Overview", level=1)                            # ---- section 7 (embed the chart)
doc.add_picture(str(chart_path), width=Inches(6.3)); doc.paragraphs[-1].alignment = WD_ALIGN_PARAGRAPH.CENTER
doc.add_heading("8. Reproducibility", level=1)                     # ---- section 8
doc.add_paragraph("The full workflow is this notebook (Restart & Run All rebuilds every output). pipeline.py "
    "runs the identical workflow head-less. The macro step calls the live World Bank API and falls back to "
    "cached raw JSON if offline, so the run is reproducible.")
doc.save(OUT_DIR / "project_memo.docx")                            # write the .docx deliverable
print("Saved overview_chart.png and project_memo.docx")

Saved overview_chart.png and project_memo.docx


**After:** the notebook ends by producing the two presentation deliverables. The chart gives an at-a-glance
view of Grameenphone's month-end price and monthly returns (the April-2020 break is visible as a gap), and the
`.docx` memo packages the company rationale, sources, the firm/macro values table, the method, the alignment
rule, the data-quality notes (including the DSE closure) and the embedded chart into a single document a reader
can open without running any code. Together with the final CSV, data dictionary, source log and quality-check
sheet, this completes every required deliverable — all regenerated from one top-to-bottom run.